In [5]:
import math
from enum import Enum, auto

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import OneHotEncoder, StandardScaler
import optuna
import json

# =========================
# 0) MDN class (your version)
# =========================
class NoiseType(Enum):
    DIAGONAL = auto()
    ISOTROPIC = auto()
    ISOTROPIC_ACROSS_CLUSTERS = auto()
    FIXED = auto()

class MixtureDensityNetwork(nn.Module):
    """
    Mixture density network (tuned for tabular)
    """
    def __init__(self, dim_in, dim_out, n_components, hidden_dim,
                 noise_type=NoiseType.DIAGONAL, fixed_noise_level=None):
        super().__init__()
        assert (fixed_noise_level is not None) == (noise_type is NoiseType.FIXED)
        num_sigma_channels = {
            NoiseType.DIAGONAL: dim_out * n_components,
            NoiseType.ISOTROPIC: n_components,
            NoiseType.ISOTROPIC_ACROSS_CLUSTERS: 1,
            NoiseType.FIXED: 0,
        }[noise_type]
        self.dim_in, self.dim_out, self.n_components = dim_in, dim_out, n_components
        self.noise_type, self.fixed_noise_level = noise_type, fixed_noise_level

        self.pi_network = nn.Sequential(
            nn.Linear(dim_in, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_components),
        )
        self.normal_network = nn.Sequential(
            nn.Linear(dim_in, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim_out * n_components + num_sigma_channels)
        )

    def forward(self, x):
        # log_pi: (bsz, K), mu: (bsz, K, D), sigma: (bsz, K, D)
        log_pi = F.log_softmax(self.pi_network(x), dim=-1)
        normal_params = self.normal_network(x)

        mu = normal_params[..., :self.dim_out * self.n_components]
        log_sigma = normal_params[..., self.dim_out * self.n_components:]

        # clamp for stability
        log_sigma = torch.clamp(log_sigma, min=-7.0, max=3.0)

        if self.noise_type is NoiseType.DIAGONAL:
            sigma = torch.exp(log_sigma)
        elif self.noise_type is NoiseType.ISOTROPIC:
            sigma = torch.exp(log_sigma).repeat(1, self.dim_out)
        elif self.noise_type is NoiseType.ISOTROPIC_ACROSS_CLUSTERS:
            sigma = torch.exp(log_sigma).repeat(1, self.n_components * self.dim_out)
        elif self.noise_type is NoiseType.FIXED:
            sigma = torch.full_like(mu, fill_value=self.fixed_noise_level)

        mu = mu.reshape(-1, self.n_components, self.dim_out)
        sigma = sigma.reshape(-1, self.n_components, self.dim_out)
        return log_pi, mu, sigma

    def loss(self, x, y):
        log_pi, mu, sigma = self.forward(x)
        z_score = (y.unsqueeze(1) - mu) / sigma

        constant = -0.5 * self.dim_out * math.log(2 * math.pi)

        normal_loglik = (
            -0.5 * torch.einsum("bij,bij->bi", z_score, z_score)
            - torch.sum(torch.log(sigma), dim=-1)
            + constant
        )
        loglik = torch.logsumexp(log_pi + normal_loglik, dim=-1)
        return -loglik.mean()

    @torch.no_grad()
    def sample(self, x, temperature=1.0):
        log_pi, mu, sigma = self.forward(x)
        cum_pi = torch.cumsum(torch.exp(log_pi), dim=-1)
        rvs = torch.rand(len(x), 1, device=x.device)
        rand_pi = torch.searchsorted(cum_pi, rvs)  # (N, 1)
        # rand_normal = torch.randn_like(mu) * sigma + mu
        rand_normal = torch.randn_like(mu) * (sigma * temperature) + mu
        samples = torch.take_along_dim(rand_normal, indices=rand_pi.unsqueeze(-1), dim=1).squeeze(dim=1)
        return samples  # (N, dim_out)
    
def batch_iter(X_data, y_data, batch_size):
    n = X_data.shape[0]
    perm = torch.randperm(n, device=X_data.device)
    for i in range(0, n, batch_size):
        j = perm[i:i+batch_size]
        yield X_data[j], y_data[j]

# =========================
# 3) Setup Optuna Objective Function
# =========================
def objective(trial):
    # Khai báo không gian siêu tham số
    n_components = trial.suggest_int("n_components", 1, 5)
    hidden_dim = trial.suggest_categorical("hidden_dim", [16, 32, 64, 128])
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    
    epochs = 300
    early_stop_patience = 30

    model = MixtureDensityNetwork(
        dim_in=X.shape[1], dim_out=1,
        n_components=n_components, hidden_dim=hidden_dim,
        noise_type=NoiseType.DIAGONAL
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_nll = float("inf")
    patience = 0
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in batch_iter(X_tr, y_tr, batch_size):
            opt.zero_grad()
            loss = model.loss(xb, yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = model.loss(X_val, y_val).item()
            
        if val_loss < best_val_nll - 1e-4:
            best_val_nll = val_loss
            patience = 0
        else:
            patience += 1
            if patience >= early_stop_patience:
                break               
        trial.report(best_val_nll, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return best_val_nll

# =========================
# 1) Config
# =========================
SEED = 121
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_PATH = r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\random-split\wpt_train_random.csv"
OUT_AUG_PATH = "wpt_train_random_aug_mdn.csv" # with synthetic rows
OUT_SYN_ONLY_PATH = "wpt_train_random_syn_only_mdn.csv" # only synthetic rows
OPTUNA_PARAMS_PATH = "best_mdn_params.json"

# Balancing policy
BALANCE_MODE = "p75"   # "max" | "p75" | "median"
MIN_GROUP_SIZE = 5
SYN_PER_TYPE_CAP = 500

# Learning config
EPOCHS = 500
EARLY_STOP_PATIENCE = 50
OPTUNA_TRIALS = 30

# Final config
FINAL_EPOCHS = 600
FINAL_PATIENCE = 50

# Control generation config
MAX_SYN_MULTIPLIER = 2  
TARGET_MINIMUM = 100     
SYN_PER_TYPE_CAP = 300 

# =========================
# 1) Load train data
# =========================
df = pd.read_csv(TRAIN_PATH)
expected = {"type", "load", "efficiency"}
missing = expected - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["type"] = df["type"].astype(str)
df["load"] = pd.to_numeric(df["load"], errors="coerce")
df["efficiency"] = pd.to_numeric(df["efficiency"], errors="coerce")
df = df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

print("Train shape:", df.shape)
print("Device:", DEVICE)

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
scaler_x = StandardScaler()
scaler_y = StandardScaler()

df_type_cond = pd.DataFrame({"type": df["type"]})
type_oh = ohe.fit_transform(df_type_cond)

df_load_cond = pd.DataFrame({"load": df["load"]})
load_scaled = scaler_x.fit_transform(df_load_cond)

X = np.hstack([type_oh, load_scaled]).astype(np.float32)
y = df[["efficiency"]].values.astype(np.float32)
y_scaled = scaler_y.fit_transform(y).astype(np.float32)

X_t = torch.tensor(X, device=DEVICE)
y_t = torch.tensor(y_scaled, device=DEVICE)

idx = np.arange(len(df))
np.random.shuffle(idx)
val_ratio = 0.10
val_n = int(len(df) * val_ratio)
val_idx, tr_idx = idx[:val_n], idx[val_n:]

X_tr, y_tr = X_t[tr_idx], y_t[tr_idx]
X_val, y_val = X_t[val_idx], y_t[val_idx]

# =========================
# 2) Chạy Optuna Optimization
# =========================
print(f"=== Starting to find Optuna ({OPTUNA_TRIALS} Trials) ===")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=OPTUNA_TRIALS)

print("\n=== Optuna Finish ===")
best_params = study.best_params
print(f"Best loss value: {study.best_value:.4f}")
print(f"Best paremeters: {best_params}")

with open(OPTUNA_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=4)
print(f"The best hyperparameter has been saved: {OPTUNA_PARAMS_PATH}")


# =========================
# 3) Train MDN with early stopping
# =========================
print("\n=== Start training the final MDN model ===")

mdn = MixtureDensityNetwork(
    dim_in=X.shape[1],
    dim_out=1,
    n_components=best_params["n_components"],
    hidden_dim=best_params["hidden_dim"],
    noise_type=NoiseType.DIAGONAL
).to(DEVICE)

opt = torch.optim.Adam(mdn.parameters(), lr=best_params["lr"])

def batch_iter(X, y, batch_size):
    n = X.shape[0]
    perm = torch.randperm(n, device=X.device)
    for i in range(0, n, batch_size):
        j = perm[i:i+batch_size]
        yield X[j], y[j]

best_val = float("inf")
best_state = None
patience = 0

for epoch in range(1, EPOCHS + 1):
    mdn.train()
    tr_losses = []
    for xb, yb in batch_iter(X_tr, y_tr, best_params["batch_size"]):
        opt.zero_grad()
        loss = mdn.loss(xb, yb)
        loss.backward()
        opt.step()
        tr_losses.append(loss.item())

    mdn.eval()
    with torch.no_grad():
        val_loss = mdn.loss(X_val, y_val).item()

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | train_nll={np.mean(tr_losses):.4f} | val_nll={val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in mdn.state_dict().items()}
        patience = 0
    else:
        patience += 1
        if patience >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val_nll={best_val:.4f}")
            break

if best_state is not None:
    mdn.load_state_dict(best_state)
mdn.eval()

# =========================
# 4) Decide target count per type (balancing)
# =========================
counts = df["type"].value_counts().sort_index()
# if BALANCE_MODE == "max":
#     target_count = int(counts.max())
# elif BALANCE_MODE == "median":
#     target_count = int(counts.median())
# elif BALANCE_MODE == "p75":
#     target_count = int(np.percentile(counts.values, 75))
# else:
#     raise ValueError("BALANCE_MODE must be 'max', 'median', or 'p75'")

print("\nCounts by type (before):\n", counts)
# print("Target per type:", target_count)

# =========================
# 5) Generate synthetic rows: efficiency* | (type, load)
#    Resample load from the same type to keep load grid + avoid bias
# =========================
syn_parts = []
eff_min, eff_max = df["efficiency"].min(), df["efficiency"].max()

for t in sorted(df["type"].unique()):
    n_real = int((df["type"] == t).sum())

    if t == 'K' or n_real > 300:
        continue

    desired_target = max(TARGET_MINIMUM, int(counts.median()))
    need = desired_target - n_real
    max_allowed_syn = int(n_real * MAX_SYN_MULTIPLIER)
    need = min(need, max_allowed_syn)
    need = min(need, SYN_PER_TYPE_CAP)

    if need <= 0:
        continue
    need = min(need, SYN_PER_TYPE_CAP)

    loads_t = df.loc[df["type"] == t, "load"].values
    loads_syn = np.random.choice(loads_t, size=need, replace=True).reshape(-1, 1)

    # build X_cond for sampling
    type_syn = np.array([t] * need).reshape(-1, 1)
    type_oh_syn = ohe.transform(type_syn)
    load_scaled_syn = scaler_x.transform(loads_syn)
    X_syn = np.hstack([type_oh_syn, load_scaled_syn]).astype(np.float32)
    X_syn_t = torch.tensor(X_syn, device=DEVICE)

    # sample y_scaled then invert scaling
    y_syn_scaled = mdn.sample(X_syn_t, temperature=0.8).detach().cpu().numpy()  # (N, 1)
    eff_syn = scaler_y.inverse_transform(y_syn_scaled).reshape(-1)
    eff_syn = np.clip(eff_syn, eff_min, eff_max)
    syn_df = pd.DataFrame({
        "type": t,
        "load": loads_syn.reshape(-1),
        "efficiency": eff_syn,
        "is_synthetic": 1
    })
    syn_parts.append(syn_df)

syn_df_all = pd.concat(syn_parts, ignore_index=True) if syn_parts else pd.DataFrame(
    columns=["type","load","efficiency","is_synthetic"]
)

real_df = df.copy()
real_df["is_synthetic"] = 0
aug_df = pd.concat([real_df, syn_df_all], ignore_index=True)

# =========================
# 7) Save augmented train + reports
# =========================
before = df["type"].value_counts().sort_index()
after = aug_df["type"].value_counts().sort_index()
report = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)
report.to_csv("mdn_type_counts_before_after.csv", encoding="utf-8-sig")

aug_df.to_csv(OUT_AUG_PATH, index=False, encoding="utf-8-sig")
if not syn_df_all.empty:
    syn_df_all.to_csv(OUT_SYN_ONLY_PATH, index=False, encoding="utf-8-sig")

print("\nGenerated synthetic rows:", len(syn_df_all))
print("Saved augmented train:", OUT_AUG_PATH)
print("Saved counts report: mdn_type_counts_before_after.csv")
print("\nCounts by type (after):\n", after)

# Optional: quick per-type mean/std check
stats_before = df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_before"})
stats_after  = aug_df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_after"})
stats_merge = stats_before.join(stats_after, lsuffix="_real", rsuffix="_aug")
stats_merge.to_csv("mdn_type_stats_before_after.csv", encoding="utf-8-sig")
print("Saved: mdn_type_stats_before_after.csv")

[I 2026-03-10 13:40:46,075] A new study created in memory with name: no-name-d6c662ff-c730-4079-8652-ff6bd83ff506


Train shape: (1310, 3)
Device: cuda
=== Starting to find Optuna (30 Trials) ===


[I 2026-03-10 13:41:14,843] Trial 0 finished with value: -1.5588129758834839 and parameters: {'n_components': 3, 'hidden_dim': 32, 'lr': 0.0025325890078671193, 'batch_size': 16}. Best is trial 0 with value: -1.5588129758834839.
[I 2026-03-10 13:41:42,017] Trial 1 finished with value: -1.5734782218933105 and parameters: {'n_components': 3, 'hidden_dim': 32, 'lr': 0.0006321309925521475, 'batch_size': 16}. Best is trial 1 with value: -1.5734782218933105.
[I 2026-03-10 13:42:05,088] Trial 2 finished with value: -1.2882959842681885 and parameters: {'n_components': 2, 'hidden_dim': 16, 'lr': 0.0005496682583301845, 'batch_size': 16}. Best is trial 1 with value: -1.5734782218933105.
[I 2026-03-10 13:42:20,844] Trial 3 finished with value: -1.4707541465759277 and parameters: {'n_components': 1, 'hidden_dim': 32, 'lr': 0.0005700543763979308, 'batch_size': 32}. Best is trial 1 with value: -1.5734782218933105.
[I 2026-03-10 13:42:34,055] Trial 4 finished with value: -1.5290971994400024 and paramet


=== Optuna Finish ===
Best loss value: -1.6160
Best paremeters: {'n_components': 3, 'hidden_dim': 32, 'lr': 0.003393139217844461, 'batch_size': 16}
The best hyperparameter has been saved: best_mdn_params.json

=== Start training the final MDN model ===
Epoch 001 | train_nll=0.2154 | val_nll=-0.1037
Epoch 005 | train_nll=-1.4173 | val_nll=-1.1394
Epoch 010 | train_nll=-1.4817 | val_nll=-1.4310
Epoch 015 | train_nll=-1.5537 | val_nll=-1.2657
Epoch 020 | train_nll=-1.5629 | val_nll=-1.3536
Epoch 025 | train_nll=-1.6090 | val_nll=-1.3249
Epoch 030 | train_nll=-1.6449 | val_nll=-1.4471
Epoch 035 | train_nll=-1.5596 | val_nll=-1.4098
Epoch 040 | train_nll=-1.6415 | val_nll=-1.5371
Epoch 045 | train_nll=-1.6311 | val_nll=-1.5068
Epoch 050 | train_nll=-1.5875 | val_nll=-1.1650
Epoch 055 | train_nll=-1.6614 | val_nll=-1.6203
Epoch 060 | train_nll=-1.7074 | val_nll=-1.5188
Epoch 065 | train_nll=-1.5376 | val_nll=-1.4578
Epoch 070 | train_nll=-1.6617 | val_nll=-1.5457
Epoch 075 | train_nll=-1.71

c:\Users\NFU_Power_Lab\anaconda3\envs\SOH\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\NFU_Power_Lab\anaconda3\envs\SOH\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\NFU_Power_Lab\anaconda3\envs\SOH\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\NFU_Power_Lab\anaconda3\envs\SOH\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\NFU_Power_Lab\anaconda3\envs\SOH\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder 